In [7]:
import torch
from torch.utils.data import DataLoader
import numpy as np
from model import TurbineFaultNet, WindTurbineDataset

print("📂 开始加载已保存的数据集...")

save_path = "data\关键中转数据\processed_turbine_data.pt"

try:
    # 1. 加载数据
    checkpoint = torch.load(save_path, weights_only=False) 
    # 注意：weights_only=False 是因为我们保存了自定义的 Dataset 对象。
    # 如果是较新版本的 PyTorch 且担心安全，确保加载环境可信。

    # 2. 提取数据集对象
    train_dataset = checkpoint['train']
    val_dataset = checkpoint['val']
    test_dataset = checkpoint['test']
    stats = checkpoint['stats']
    
    print("\n✅ 数据加载成功！")
    
    # 3. 打印统计信息 (验证数据完整性)
    print(f"   - 信号形状记录: {stats['signal_shape']}")
    print(f"   - 特征形状记录: {stats['feature_shape']}")
    print(f"   - 类别分布: {stats['class_distribution']}")
    
    total_samples = len(train_dataset) + len(val_dataset) + len(test_dataset)
    print(f"   - 恢复总样本数: {total_samples}")

    # 4. 重新构建 DataLoader
    # 这里的 batch_size 可以根据你的显存情况重新调整，不需要和保存时一致
    batch_size = 32 
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    print(f"\n💾 DataLoader 重建完成 (Batch Size = {batch_size}):")

    # 5. 快速验证数据可用性 (Dry Run)
    print("\n🧪 正在验证数据批次...")
    sample_batch = next(iter(train_loader))
    
    assert 'signal' in sample_batch, "错误：Batch 中缺少 'signal' 键"
    assert 'feature' in sample_batch, "错误：Batch 中缺少 'feature' 键"
    assert 'label' in sample_batch, "错误：Batch 中缺少 'label' 键"
    
    print(f"   - Signal Batch Shape:  {sample_batch['signal'].shape}")   # 应为 [32, 3, 102400]
    print(f"   - Feature Batch Shape: {sample_batch['feature'].shape}")  # 应为 [32, 3, 12]
    print(f"   - Label Batch Shape:   {sample_batch['label'].shape}")    # 应为 [32]
    print(f"   - Label 示例值:        {sample_batch['label'][:5].numpy()}")
    
    print("\n🎉 数据集准备就绪！可以直接传入模型训练。")

except FileNotFoundError:
    print(f"❌ 错误：未找到文件 {save_path}，请先运行保存脚本。")
except Exception as e:
    print(f"❌ 加载过程中发生错误：{e}")
    print("提示：如果 PyTorch 版本差异较大，可能需要重新运行原始脚本来生成数据。")

📂 开始加载已保存的数据集...

✅ 数据加载成功！
   - 信号形状记录: (439, 3, 102400)
   - 特征形状记录: (439, 3, 12)
   - 类别分布: [147 292]
   - 恢复总样本数: 439

💾 DataLoader 重建完成 (Batch Size = 32):

🧪 正在验证数据批次...
   - Signal Batch Shape:  torch.Size([32, 3, 102400])
   - Feature Batch Shape: torch.Size([32, 3, 12])
   - Label Batch Shape:   torch.Size([32])
   - Label 示例值:        [1 1 1 1 1]

🎉 数据集准备就绪！可以直接传入模型训练。


In [8]:
try:
    batch = next(iter(test_loader))
except StopIteration:
    raise ValueError("❌ 错误：测试集 DataLoader 为空！请检查数据集划分。")

signal_batch = batch['signal']
feature_batch = batch['feature']
label_batch = batch['label']


In [9]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import os
import time

# 从 model.py 导入 (请确保 model.py 在当前目录或路径正确)
# 如果是在同一个文件中，请注释掉下面这行并使用本地定义的类
from model import WindTurbineDataset, TurbineFaultNet 

# ==========================================
# 1. 定义 PyTorch Lightning 模块 (增强版)
# ==========================================
class TurbineFaultLitModel(pl.LightningModule):
    def __init__(self, model_config, learning_rate=1e-3, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = TurbineFaultNet(**model_config)
        self.criterion = torch.nn.CrossEntropyLoss()
        
        # 用于存储当前 epoch 的指标
        self.training_step_outputs = []
        self.validation_step_outputs = []

    def forward(self, signal, feature, turbine_id=None):
        return self.model(signal, feature, turbine_id)

    def training_step(self, batch, batch_idx):
        signal, feature, label = batch['signal'], batch['feature'], batch['label']
        logits = self(signal, feature, turbine_id=None)
        loss = self.criterion(logits, label)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == label).float().mean()
        
        # 记录单个 step 的结果供 epoch 结束时汇总
        self.training_step_outputs.append({'loss': loss, 'acc': acc})
        
        # 实时记录 (用于 TensorBoard 和 tqdm，虽然我们要关掉 tqdm)
        self.log('train_loss_step', loss, on_step=True, on_epoch=False, prog_bar=False, logger=True)
        return loss

    def on_train_epoch_end(self):
        """每个训练 Epoch 结束时调用"""
        if not self.training_step_outputs:
            return
            
        # 计算平均值
        avg_loss = torch.stack([x['loss'] for x in self.training_step_outputs]).mean()
        avg_acc = torch.stack([x['acc'] for x in self.training_step_outputs]).mean()
        
        # 记录到 logger
        self.log('train_loss', avg_loss, on_epoch=True, prog_bar=False, logger=True, sync_dist=True)
        self.log('train_acc', avg_acc, on_epoch=True, prog_bar=False, logger=True, sync_dist=True)
        
        # 🖨️ 【关键修改】打印详细日志
        current_lr = self.trainer.optimizers[0].param_groups[0]['lr']
        print(f"\n[Epoch {self.current_epoch:03d}] "
              f"Train Loss: {avg_loss.item():.4f} | "
              f"Train Acc : {avg_acc.item():.4f} | "
              f"LR: {current_lr:.6f}", flush=True)
        
        # 清空列表以便下一个 epoch 使用
        self.training_step_outputs.clear()

    def validation_step(self, batch, batch_idx):
        signal, feature, label = batch['signal'], batch['feature'], batch['label']
        logits = self(signal, feature, turbine_id=None)
        loss = self.criterion(logits, label)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == label).float().mean()
        
        self.validation_step_outputs.append({'val_loss': loss, 'val_acc': acc})
        return {'val_loss': loss, 'val_acc': acc}

    def on_validation_epoch_end(self):
        """每个验证 Epoch 结束时调用"""
        if not self.validation_step_outputs:
            return

        avg_loss = torch.stack([x['val_loss'] for x in self.validation_step_outputs]).mean()
        avg_acc = torch.stack([x['val_acc'] for x in self.validation_step_outputs]).mean()
        
        self.log('val_loss', avg_loss, on_epoch=True, prog_bar=False, logger=True, sync_dist=True)
        self.log('val_acc', avg_acc, on_epoch=True, prog_bar=False, logger=True, sync_dist=True)
        
        # 🖨️ 【关键修改】打印详细日志
        print(f"           Val  Loss: {avg_loss.item():.4f} | "
              f"Val  Acc : {avg_acc.item():.4f} "
              f"{'⭐' if avg_acc > 0.9 else ''}", flush=True) # 如果准确率高加个星星
        
        # 清空列表
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        signal, feature, label = batch['signal'], batch['feature'], batch['label']
        logits = self(signal, feature, turbine_id=None)
        loss = self.criterion(logits, label)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == label).float().mean()
        
        self.log('test_loss', loss, on_epoch=True, logger=True)
        self.log('test_acc', acc, on_epoch=True, logger=True)
        return {'preds': preds, 'labels': label}

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss", "interval": "epoch"}}

# ==========================================
# 2. 主训练流程
# ==========================================
def main():
    # 🖨️ 打印训练表头
    print("="*60)
    print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Train Acc':<12} | {'Val Loss':<12} | {'Val Acc':<12} | LR")
    print("-"*60)
    
    # --- 配置 ---
    DATA_PATH = r"data\关键中转数据\processed_turbine_data.pt"
    BATCH_SIZE = 8
    NUM_WORKERS = 0  # Windows/Jupyter 建议设为 0，Linux 服务器可改为 CPU 核心数
    MAX_EPOCHS = 50
    LEARNING_RATE = 1e-3
    NUM_TURBINES = 8
    
    model_config = {
        "num_classes": 2,
        "d_model": 32,
        "num_turbines": NUM_TURBINES
        # 注意：这里不再传递 stat_total_dim，因为模型内部已硬编码或自动处理
    }

    # --- 1. 加载数据 ---
    if not os.path.exists(DATA_PATH):
        # 为了演示代码运行，如果没有文件，这里生成随机数据 (实际使用时请删除此块)
        print(f"⚠️ 未找到数据文件 {DATA_PATH}，正在生成随机演示数据...")
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        N = 200
        signals = np.random.randn(N, 3, 2048).astype(np.float32)
        features = np.random.randn(N, 3, 11).astype(np.float32)
        ids = np.random.randint(0, 8, (N, 1)).astype(np.float32)
        features = np.concatenate([features, ids], axis=-1) # 拼接 ID
        labels = np.random.randint(0, 2, N)
        torch.save({'signals': signals, 'features': features, 'labels': labels}, DATA_PATH)
    
    print(f"📂 加载数据：{DATA_PATH}")
    try:
        raw_data = torch.load(DATA_PATH, map_location='cpu', weights_only=False)
    except Exception:
        raw_data = np.load(DATA_PATH, allow_pickle=True).item()

    # 适配可能的数据结构
    if isinstance(raw_data, dict):
        signals = raw_data.get('signals', raw_data.get('signal'))
        features = raw_data.get('features', raw_data.get('feature'))
        labels = raw_data.get('labels', raw_data.get('label'))
    else:
        raise ValueError("数据格式无法解析")



    # --- 创建 DataLoader ---
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    
    print(f"✅ 数据准备完成 | Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    # --- 数据检查 (简化版，避免干扰日志) ---
    print("\n🛡️ 快速数据检查...")
    batch = next(iter(train_loader))
    assert batch['feature'].shape[-1] == 12, "特征维度必须为 12"
    print("   ✅ 检查通过\n")

    # --- 2. 初始化模型与训练 ---
    lit_model = TurbineFaultLitModel(model_config=model_config, learning_rate=LEARNING_RATE)
    
    checkpoint_callback = ModelCheckpoint(
        monitor='val_acc', 
        dirpath='checkpoints/', 
        filename='best-{epoch:02d}-{val_acc:.4f}', 
        save_top_k=3, 
        mode='max',
        save_last=True
    )
    early_stop_callback = EarlyStopping(monitor='val_acc', patience=15, mode='max', verbose=False)
    logger = TensorBoardLogger(save_dir="logs/", name="turbine_fault")

    # 🚀 Trainer 配置修改
    trainer = pl.Trainer(
        accelerator='auto', 
        devices=1, 
        max_epochs=MAX_EPOCHS,
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=logger, 
        log_every_n_steps=1,          # 每个 step 都记录
        gradient_clip_val=1.0,
        precision='16-mixed' if torch.cuda.is_available() else '32',
        enable_progress_bar=False,    # 🔴 关闭默认进度条，防止覆盖 print 输出
        enable_model_summary=False,   # 可选：关闭初始模型摘要，保持输出整洁
        check_val_every_n_epoch=1     # 每个 epoch 都验证
    )

    print("🚀 开始训练...\n")
    start_time = time.time()
    
    try:
        trainer.fit(lit_model, train_loader, val_loader)
    except KeyboardInterrupt:
        print("\n⚠️ 用户中断训练")
    
    end_time = time.time()
    print(f"\n⏱️ 训练总耗时: {end_time - start_time:.2f} 秒")
    print(f"💾 最佳模型路径：{checkpoint_callback.best_model_path}")

    # --- 测试 ---
    if test_loader:
        print("\n🧪 开始在测试集上评估...")
        # 加载最佳模型进行测试
        best_model = TurbineFaultLitModel.load_from_checkpoint(checkpoint_callback.best_model_path, model_config=model_config, learning_rate=LEARNING_RATE)
        trainer.test(best_model, dataloaders=test_loader)

if __name__ == "__main__":
    import matplotlib
    matplotlib.use('Agg') 
    main()

Epoch    | Train Loss   | Train Acc    | Val Loss     | Val Acc      | LR
------------------------------------------------------------
📂 加载数据：data\关键中转数据\processed_turbine_data.pt


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


✅ 数据准备完成 | Train: 219, Val: 109, Test: 111

🛡️ 快速数据检查...
   ✅ 检查通过

🚀 开始训练...

           Val  Loss: 0.6972 | Val  Acc : 0.3750 
           Val  Loss: 0.3111 | Val  Acc : 0.8696 

[Epoch 000] Train Loss: 0.5097 | Train Acc : 0.7530 | LR: 0.000976
           Val  Loss: 0.3489 | Val  Acc : 0.8964 

[Epoch 001] Train Loss: 0.3649 | Train Acc : 0.8705 | LR: 0.000905
           Val  Loss: 0.0884 | Val  Acc : 0.9911 ⭐

[Epoch 002] Train Loss: 0.2809 | Train Acc : 0.8795 | LR: 0.000794
           Val  Loss: 0.1043 | Val  Acc : 0.9768 ⭐

[Epoch 003] Train Loss: 0.3181 | Train Acc : 0.8899 | LR: 0.000655
           Val  Loss: 0.0352 | Val  Acc : 1.0000 ⭐

[Epoch 004] Train Loss: 0.1773 | Train Acc : 0.9375 | LR: 0.000501
           Val  Loss: 0.0269 | Val  Acc : 1.0000 ⭐

[Epoch 005] Train Loss: 0.2150 | Train Acc : 0.9330 | LR: 0.000346
           Val  Loss: 0.0212 | Val  Acc : 1.0000 ⭐

[Epoch 006] Train Loss: 0.1499 | Train Acc : 0.9554 | LR: 0.000207
           Val  Loss: 0.0494 | Val  Acc 

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



⏱️ 训练总耗时: 36.27 秒
💾 最佳模型路径：E:\Grad\checkpoints\best-epoch=04-val_acc=1.0000-v4.ckpt

🧪 开始在测试集上评估...
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc                    1.0
        test_loss           0.04518800973892212
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
